# Anime forum sentiment · 5,000 comments per title
Collect AniList forum discussions separately from MAL reviews. CPU is enough; a GPU is not required. Actual totals may be smaller than the cap. Sentiment is predicted by VADER, not validated against human labels.

Run the cells in order. The notebook uses the pull-request branch until it is merged. Change `BRANCH` to `main` after merging.

In [11]:
BRANCH = "design/anime-sentiment-atlas"
LIMIT = 5000
import subprocess, os
from pathlib import Path
repo = Path("/content/anime-reviews")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/harsh-github007/Anime-Reviews-Sentimental-Analysis.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

CompletedProcess(args=['python', '-m', 'pip', 'install', '-e', '.'], returncode=0)

## Optional: keep checkpoints in Google Drive
Enable this before collecting to resume after a Colab disconnect. Re-running the collection resumes saved checkpoints. Raw comments stay in your own Drive; only aggregate results are used by the website.

In [12]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    checkpoint_dir = "/content/drive/MyDrive/AnimeForum/checkpoints"
else:
    checkpoint_dir = "data/forum"
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
# Reuse the locally collected 12,337-comment dataset uploaded to your Drive.
import zipfile
archive = Path("/content/drive/MyDrive/AnimeForum/forum-checkpoints.zip")
if USE_DRIVE and archive.exists():
    with zipfile.ZipFile(archive) as bundle:
        for name in bundle.namelist():
            if Path(name).name != name or not name.endswith(".json"):
                raise ValueError("Unexpected checkpoint archive entry")
            target = Path(checkpoint_dir) / name
            incoming = json.loads(bundle.read(name))
            existing = json.loads(target.read_text()) if target.exists() else {}
            if len(incoming.get("comments", [])) >= len(existing.get("comments", [])):
                target.write_text(json.dumps(incoming, ensure_ascii=False))
    print("Imported Drive checkpoints; completed titles will be reused.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imported Drive checkpoints; completed titles will be reused.


In [13]:
subprocess.run(["python", "-m", "animesent", "forum", "--limit", str(LIMIT), "--data", checkpoint_dir, "--out", "results/forum.json"], check=True)

CompletedProcess(args=['python', '-m', 'animesent', 'forum', '--limit', '5000', '--data', '/content/drive/MyDrive/AnimeForum/checkpoints', '--out', 'results/forum.json'], returncode=0)

## MyAnimeList coverage without an API key
The collector reads public anime-linked forum pages if no Client ID is configured. It stops on blocked or unrecognised pages and reports partial coverage. No GPU is needed. Comments are checkpointed per title. A Client ID remains optional for using the official API.

## Inspect actual counts and collection status
A title is complete when its available discussions are exhausted or the cap is reached. If errors appear, rerun the collection cell to retry. No automatic GitHub upload occurs.

In [14]:
import json
import pandas as pd
summary = json.loads(Path("results/forum.json").read_text())
rows = [{"Anime": a["title"], "Comments": a["comments"], "Complete": a["complete"], "Error": a["error"], **a["counts"]} for a in summary["anime"]]
display(pd.DataFrame(rows))
print("Total comments:", sum(a["comments"] for a in summary["anime"]))

,Anime,Comments,Complete,Error,Positive,Neutral,Negative
0,Steel Ball Run: JoJo no Kimyou na Bouken,510,True,None,219,185,106
1,Bleach: Sennen Kessen-hen - Kashin-tan,363,True,None,166,89,108
2,One Piece,5000,True,None,2590,1461,949
3,Chiikawa,7,True,None,2,4,1
4,Shiguang Dailiren III,89,True,None,44,20,25
5,Seihantai na Kimi to Boku 2nd Season,347,True,None,220,64,63
6,Xian Ni,1,True,None,0,1,0
7,Tian Guan Cifu Short Films,0,True,None,0,0,0
8,Doupo Cangqiong: Nian Fan,16,True,None,10,4,2
9,Tunshi Xingkong 4th Season,0,True,None,0,0,0


Total comments: 6333


In [ ]:
from google.colab import files
files.download("results/forum.json")